In [2]:
import re
import spacy
from collections import defaultdict
import Levenshtein as lev

In [6]:
!python -m spacy download es_core_news_sm

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     --------------------------------------- 0.0/12.9 MB 131.3 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.9 MB 178.6 kB/s eta 0:01:12
     --------------------------------------- 0.1/12.9 MB 252.2 kB/s eta 0:00:51
     --------------------------------------- 0.1/12.9 MB 425.1 kB/s eta 0:00:31
      -------------------------------------- 0.2/12.9 MB 724.0 kB/s eta 0:00:18
     - -------------------------------------- 0.5/12.9 MB 1.3 MB/s eta 0:00:10
     --- ------------------------------------ 1.0/12.9 MB 2.6 MB/s eta 0:00:05
     ----- ---------------------------------- 1.7/12.9 MB 3.8 MB/s eta 0:00:03
     ------ --------------------------------- 2.2/12.9 MB 4.6 MB/s eta 0:00:03
     -------- ------------------------------- 2.8/12.9 MB 5.3 MB/s e

In [3]:
# Cargar modelo de español de spaCy (necesitas: python -m spacy download es_core_news_sm)
try:
    nlp = spacy.load("es_core_news_sm")
except:
    print("Instala el modelo: python -m spacy download es_core_news_sm")
    nlp = None

In [ ]:
class ResolucionCorreferencias:
    """
    Sistema de resolución de correferencias para textos en español.
    Identifica y vincula referencias pronominales y nominales a sus antecedentes.
    """
    def __init__(self):
        self.entidades = []  # Lista de entidades detectadas
        self.correferencias = defaultdict(list)  # Mapeo de referencias
        self.pronombres = {
            'él', 'ella', 'ellos', 'ellas', 'lo', 'la', 'los', 'las',
            'le', 'les', 'se', 'su', 'sus', 'este', 'esta', 'estos', 
            'estas', 'ese', 'esa', 'esos', 'esas', 'aquel', 'aquella',
            'aquellos', 'aquellas'
        }
        
    def extraer_entidades(self, texto):
        """
        Extrae entidades nombradas usando spaCy y patrones personalizados.
        Retorna lista de entidades con su posición y tipo.
        """
        if not nlp:
            return []
            
        doc = nlp(texto)
        entidades = []
        
        # Entidades de spaCy (PER, ORG, LOC, etc.)
        for ent in doc.ents:
            entidades.append({
                'texto': ent.text,
                'inicio': ent.start_char,
                'fin': ent.end_char,
                'tipo': ent.label_,
                'genero': self._inferir_genero(ent.text),
                'numero': self._inferir_numero(ent.text)
            })
        
        # Patrones adicionales para entidades complejas
        # Ejemplo: "La Abogacía del Estado", "El PP", "El PSOE"
        patrones = [
            r'\b(El|La|Los|Las)\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(\s+(del|de|de la|de los)\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)*',
            r'\b[A-ZÁÉÍÓÚÑ]{2,}\b',  # Siglas
        ]
        
        for patron in patrones:
            for match in re.finditer(patron, texto):
                texto_ent = match.group()
                # Evitar duplicados
                if not any(e['inicio'] == match.start() for e in entidades):
                    entidades.append({
                        'texto': texto_ent,
                        'inicio': match.start(),
                        'fin': match.end(),
                        'tipo': 'CUSTOM',
                        'genero': self._inferir_genero(texto_ent),
                        'numero': self._inferir_numero(texto_ent)
                    })
        
        return sorted(entidades, key=lambda x: x['inicio'])
    
    def _inferir_genero(self, texto):
        """Infiere género gramatical basado en terminaciones y artículos."""
        texto_lower = texto.lower()
        
        if any(texto_lower.startswith(art) for art in ['el ', 'los ', 'un ', 'unos ']):
            return 'M'
        if any(texto_lower.startswith(art) for art in ['la ', 'las ', 'una ', 'unas ']):
            return 'F'
        
        # Heurísticas por terminación
        if texto_lower.endswith(('o', 'os')):
            return 'M'
        if texto_lower.endswith(('a', 'as', 'ión', 'ía', 'dad', 'tad')):
            return 'F'
            
        return 'N'  # Neutro/desconocido
    
    def _inferir_numero(self, texto):
        """Infiere número gramatical."""
        texto_lower = texto.lower()
        
        if any(texto_lower.startswith(art) for art in ['los ', 'las ', 'unos ', 'unas ']):
            return 'P'
        if texto_lower.endswith(('s', 'es')) and not texto_lower.endswith(('és', 'ás', 'ís', 'ós')):
            return 'P'
            
        return 'S'
    
    def detectar_referencias(self, texto):
        """
        Detecta pronombres y expresiones referenciales en el texto.
        """
        if not nlp:
            return []
            
        doc = nlp(texto)
        referencias = []
        
        for token in doc:
            # Pronombres personales, demostrativos, posesivos
            if token.lower_ in self.pronombres or token.pos_ == 'PRON':
                referencias.append({
                    'texto': token.text,
                    'inicio': token.idx,
                    'fin': token.idx + len(token.text),
                    'tipo': token.pos_,
                    'lema': token.lemma_
                })
            
            # Determinantes definidos que pueden indicar correferencia
            elif token.pos_ == 'DET' and token.lower_ in ['el', 'la', 'los', 'las']:
                # Verificar si es parte de una entidad compuesta
                if token.i + 1 < len(doc):
                    siguiente = doc[token.i + 1]
                    if siguiente.pos_ == 'NOUN':
                        referencias.append({
                            'texto': f"{token.text} {siguiente.text}",
                            'inicio': token.idx,
                            'fin': siguiente.idx + len(siguiente.text),
                            'tipo': 'DET_NOUN',
                            'lema': siguiente.lemma_
                        })
        
        return referencias
    
    def resolver_correferencias(self, texto, ventana=3):
        """
        Resuelve correferencias vinculando referencias a sus antecedentes.
        
        Args:
            texto: Texto a analizar
            ventana: Número de oraciones previas a considerar para buscar antecedentes
            
        Returns:
            Dict con las cadenas de correferencia detectadas
        """
        # Extraer entidades y referencias
        entidades = self.extraer_entidades(texto)
        referencias = self.detectar_referencias(texto)
        
        # Dividir en oraciones para contexto
        if nlp:
            doc = nlp(texto)
            oraciones = [sent.text for sent in doc.sents]
        else:
            oraciones = re.split(r'[.!?]+', texto)
        
        cadenas_correferencia = defaultdict(list)
        
        # Para cada referencia, buscar antecedente más probable
        for ref in referencias:
            mejor_antecedente = None
            mejor_puntuacion = 0
            
            # Buscar entre entidades previas
            for ent in entidades:
                if ent['fin'] < ref['inicio']:  # Solo antecedentes
                    puntuacion = self._calcular_compatibilidad(ref, ent, texto)
                    
                    if puntuacion > mejor_puntuacion:
                        mejor_puntuacion = puntuacion
                        mejor_antecedente = ent
            
            # Si encontramos antecedente con suficiente confianza
            if mejor_antecedente and mejor_puntuacion > 0.3:
                clave = mejor_antecedente['texto']
                cadenas_correferencia[clave].append({
                    'referencia': ref['texto'],
                    'posicion': ref['inicio'],
                    'confianza': mejor_puntuacion
                })
        
        return dict(cadenas_correferencia)
    
    def _calcular_compatibilidad(self, referencia, entidad, texto):
        """
        Calcula compatibilidad entre referencia y entidad candidata.
        Usa múltiples heurísticas ponderadas.
        """
        puntuacion = 0.0
        
        # 1. Distancia (referencias cercanas más probables)
        distancia = referencia['inicio'] - entidad['fin']
        if distancia < 100:
            puntuacion += 0.3
        elif distancia < 300:
            puntuacion += 0.2
        elif distancia < 500:
            puntuacion += 0.1
        
        # 2. Similitud léxica (usando Levenshtein)
        ref_texto = referencia.get('lema', referencia['texto']).lower()
        ent_texto = entidad['texto'].lower()
        
        distancia_lev = lev.distance(ref_texto, ent_texto)
        if distancia_lev <= 2:
            puntuacion += 0.4
        elif distancia_lev <= 4:
            puntuacion += 0.2
        
        # 3. Concordancia de género (si es pronombre)
        if referencia['tipo'] == 'PRON':
            ref_lower = referencia['texto'].lower()
            if entidad['genero'] == 'M' and ref_lower in ['él', 'lo', 'le', 'su', 'este', 'ese']:
                puntuacion += 0.3
            elif entidad['genero'] == 'F' and ref_lower in ['ella', 'la', 'le', 'su', 'esta', 'esa']:
                puntuacion += 0.3
        
        # 4. Concordancia de número
        ref_lower = referencia['texto'].lower()
        if entidad['numero'] == 'P' and ref_lower in ['los', 'las', 'les', 'ellos', 'ellas', 'estos', 'estas']:
            puntuacion += 0.2
        elif entidad['numero'] == 'S' and ref_lower in ['el', 'la', 'le', 'él', 'ella', 'este', 'esta']:
            puntuacion += 0.2
        
        return min(puntuacion, 1.0)
    
    def generar_reporte(self, texto):
        """
        Genera un reporte completo de correferencias en formato legible.
        """
        correferencias = self.resolver_correferencias(texto)
        
        reporte = []
        reporte.append("=" * 60)
        reporte.append("ANÁLISIS DE CORREFERENCIAS")
        reporte.append("=" * 60)
        
        if not correferencias:
            reporte.append("\nNo se detectaron cadenas de correferencia.")
            return "\n".join(reporte)
        
        for i, (entidad, referencias) in enumerate(correferencias.items(), 1):
            reporte.append(f"\n[Cadena {i}] ENTIDAD: {entidad}")
            reporte.append("-" * 60)
            for ref in referencias:
                reporte.append(f"  → '{ref['referencia']}' (pos: {ref['posicion']}, "
                             f"confianza: {ref['confianza']:.2f})")
        
        reporte.append("\n" + "=" * 60)
        return "\n".join(reporte)


In [ ]:
if __name__ == "__main__":
    # Texto de ejemplo
    texto_ejemplo = """
    Moreno intenta apaciguar el flanco sanitario mientras enreda con la fecha de las elecciones.
    La Abogacía del Estado se retira como acusación en el caso Tándem del 'excomisario' José Villarejo.
    Las promesas incumplidas de Pablo Echenique en sanidad, educación y vivienda le pasan factura.
    Sánchez defiende resolver el problema de la ley del 'solo sí es sí' desde el diálogo.
    El PP se rinde al discurso antiabortista de Vox en pleno año electoral.
    """
    
    resolver = ResolucionCorreferencias()
    print(resolver.generar_reporte(texto_ejemplo))
    
    # Obtener las correferencias como diccionario
    correfs = resolver.resolver_correferencias(texto_ejemplo)
    print("\n\nDiccionario de correferencias:")
    for entidad, refs in correfs.items():
        print(f"{entidad}: {[r['referencia'] for r in refs]}")

ANÁLISIS DE CORREFERENCIAS

[Cadena 1] ENTIDAD: José Villarejo
------------------------------------------------------------
  → 'le' (pos: 283, confianza: 0.80)

[Cadena 2] ENTIDAD: Pablo Echenique
------------------------------------------------------------
  → 'la' (pos: 346, confianza: 0.50)

[Cadena 3] ENTIDAD: PP
------------------------------------------------------------
  → 'se' (pos: 401, confianza: 0.70)



Diccionario de correferencias:
José Villarejo: ['le']
Pablo Echenique: ['la']
PP: ['se']
